# Week 02 - Data Characteristics & Wrangling

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PASSENGER_FILE = "passengerData.csv"
TICKET_FILE    = "ticketPrices.xlsx"
TITANIC_FILE   = "titanic.csv"
TB_FILE        = "TB_burden_countries.csv"

## Part 0 - Jupyter Notebooks

In [ ]:
# Week 1 Part 1


## Part 1 - Pandas basics (tutorial code)
### Series

In [ ]:
# A Series is like one column of a table, with an index (labels) for each item
s = pd.Series([7, 'Heisenberg', 3.14, -1789710578, 'Happy Eating!'])
print(s)

# Same Series but with our own index labels
s = pd.Series([7, 'Heisenberg', 3.14, -1789710578, 'Happy Eating!'],
              index=['A', 'Z', 'C', 'Y', 'E'])
print(s)
print(s.iloc[0])   # access by position
print(s['Z'])      # access by label

In [ ]:
# Series from a dictionary: keys become the index
d = {'Chicago': 1000, 'New York': 1300, 'Portland': 900, 'San Francisco': 1100,
     'Austin': 450, 'Boston': None}
cities = pd.Series(d)
print(cities)
print(cities.isnull())   # True where a value is missing (Boston)

### DataFrames

In [ ]:
data = {'year':   [2010, 2011, 2012, 2011, 2012, 2010, 2011, 2012],
        'team':   ['Bears', 'Bears', 'Bears', 'Packers', 'Packers', 'Lions', 'Lions', 'Lions'],
        'wins':   [11, 8, 10, 15, 11, 6, 10, 4],
        'losses': [5, 8, 6, 1, 5, 10, 6, 12]}
football = pd.DataFrame(data, columns=['year', 'team', 'wins', 'losses'])
print(football)
print("Access by index ver 1", football['losses'])
print("Access by index ver 2", football.losses)

In [ ]:
# Plot with pandas data + matplotlib
plt.scatter(football.losses, football.wins)
plt.xlabel("losses")
plt.ylabel("wins")
plt.title("Football: wins vs losses")
plt.show()

In [ ]:
# Basic information about the data
print("Minimum wins:", football['wins'].min())
print(football.describe())   # summary stats (only numeric columns)
print(football.dtypes)       # data type of each column

# Modify the data (on a copy)
football_copy = football.copy()
football_copy['wins'] = 7
print(football_copy)

### Slicing and filtering

In [ ]:
print(football[2:5])                 # rows 2,3,4 (3rd to 5th row)
print(football[['year', 'wins']])    # only some columns
print(football.iloc[[3, 1]])         # rows at position 3 and 1
print(football['year'][2:5])         # rows 2-4 of the year column only

In [ ]:
print(football[football.year > 2011])                              # 2012 onwards
print(football[(football.wins > 5) & (football.losses < 5)])       # & = AND, | = OR

### Merging (like SQL joins)

In [ ]:
left_frame = pd.DataFrame({'key': range(5),
                           'left_value': ['a', 'b', 'c', 'd', 'e']})
right_frame = pd.DataFrame({'key': range(2, 7),
                            'right_value': ['f', 'g', 'h', 'i', 'j']})
print(left_frame)
print(right_frame)

print("--- An inner join:")
print(pd.merge(left_frame, right_frame, on='key'))   # only keys in BOTH (2,3,4)

new_row = pd.DataFrame({'key': [2], 'left_value': ['nv']})
left_frame_new = pd.concat([left_frame, new_row], ignore_index=True)
print(pd.merge(left_frame_new, right_frame, on='key'))  # key 2 now appears twice

print("--- Now a left outer join:")
print(pd.merge(left_frame, right_frame, on='key', how='left'))
# NaN appears for keys 0 and 1: they exist in left_frame but have no match
# in right_frame, so pandas has no right_value to put there.

## DIY Exercises - Part 1

In [ ]:
# 1. Load both files into DataFrames
passengers = pd.read_csv(PASSENGER_FILE)
tickets = pd.read_excel(TICKET_FILE)  

print(passengers.shape)                  # should be (891, number_of_columns)
print(passengers.head())
print(tickets.head())
print(passengers.columns.tolist())
print(tickets.columns.tolist())

In [ ]:
# 2. Merge on the shared column 'TicketType'
merged = pd.merge(passengers, tickets, on='TicketType')
print(merged.shape)
print(merged.head())

# The price column is whichever column came from the ticket file (not TicketType)
price_col = [c for c in tickets.columns if c != 'TicketType'][0]
print("Ticket price column is:", price_col)

In [ ]:
# 3. Name of the oldest passenger(s)
max_age = merged['Age'].max()               # intermediate variable
oldest = merged[merged['Age'] == max_age]   # all rows with that age (could be >1)
print("Oldest age:", max_age)
print(oldest['Name'])

In [ ]:
# 4. Scatter plot: Age vs Ticket Price
plt.scatter(merged['Age'], merged[price_col], alpha=0.5)
plt.xlabel("Age")
plt.ylabel("Ticket price (GBP)")
plt.title("Age vs Ticket Price")
plt.show()

In [ ]:
# 5. Female passengers aged 40-50 who paid >= 40
subset = merged[(merged['Sex'] == 'female') &
                (merged['Age'] >= 40) & (merged['Age'] <= 50) &
                (merged[price_col] >= 40)]
print(len(subset), "passengers match")

plt.scatter(subset['Age'], subset[price_col], color='purple')
plt.xlabel("Age")
plt.ylabel("Ticket price (GBP)")
plt.title("Females aged 40-50 paying >= 40")
plt.show()

## Part 2 - Missing values (tutorial code)

In [ ]:
df = pd.DataFrame(np.random.randn(5, 3), index=['a', 'c', 'e', 'f', 'h'],
                  columns=['one', 'two', 'three'])
df['four'] = 'bar'
df['five'] = df['one'] > 0
# reindex adds rows b, d, g that have no data -> they become NaN
df2 = df.reindex(['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h'])
print(df2)
print(pd.isnull(df2['one']))

## DIY Exercises - 2: Missing values

In [ ]:
# 1. Load the Titanic data
titanic = pd.read_csv(TITANIC_FILE)
print(titanic.head())

# 2. Count missing values in each column
print(titanic.isnull().sum())

In [ ]:
# 3. Mean and other descriptive statistics (write these down!)
print(titanic.describe())

In [ ]:
# 4. Replace missing Age and Fare with 0, then scatter plot
titanic_zero = titanic.copy()
titanic_zero[['Age', 'Fare']] = titanic_zero[['Age', 'Fare']].fillna(0)

plt.scatter(titanic_zero['Age'], titanic_zero['Fare'], alpha=0.5)
plt.xlabel("Age"); plt.ylabel("Fare")
plt.title("Missing values filled with 0")
plt.show()
print(titanic_zero[['Age', 'Fare']].describe())

In [ ]:
# 5. Replace missing Age and Fare with the column mean, then scatter plot
titanic_mean = titanic.copy()
titanic_mean['Age'] = titanic_mean['Age'].fillna(titanic_mean['Age'].mean())
titanic_mean['Fare'] = titanic_mean['Fare'].fillna(titanic_mean['Fare'].mean())

plt.scatter(titanic_mean['Age'], titanic_mean['Fare'], alpha=0.5, color='orange')
plt.xlabel("Age"); plt.ylabel("Fare")
plt.title("Missing values filled with the mean")
plt.show()
print(titanic_mean[['Age', 'Fare']].describe())

### 6. Reflection
- **Filling with 0:** a column of fake points appears at Age = 0 (and Fare = 0). These look like
  newborns who travelled for free, which is wrong. The mean and std of Age are pulled DOWN.
- **Filling with the mean:** a vertical line of points appears at the mean age (~29.7). The mean
  stays the same, but the spread (std) shrinks, because many identical values were added.
- Neither is perfect: both invent data. Mean-filling is usually less damaging than 0, but better
  options include the median, filling by group (e.g. mean age per passenger class), or dropping rows.

## DIY Exercises - 3: Data Transformations

In [ ]:
# 1. Load the WHO Tuberculosis data
tb = pd.read_csv(TB_FILE)
print(tb.shape)
print(tb.columns.tolist())

# 2. Replace missing values in numeric columns (here: with each column's median)
numeric_cols = tb.select_dtypes(include='number').columns
print(tb[numeric_cols].isnull().sum())
tb[numeric_cols] = tb[numeric_cols].fillna(tb[numeric_cols].median())

In [ ]:
# 3. Histograms of a few columns with different shapes
cols_to_plot = ['e_prev_100k_hi', 'e_inc_100k', 'e_pop_num']
cols_to_plot = [c for c in cols_to_plot if c in tb.columns]

for col in cols_to_plot:
    plt.hist(tb[col], bins=40)
    plt.title(f"{col} (original)")
    plt.xlabel(col); plt.ylabel("count")
    plt.show()

In [ ]:
# 4. Log transformation, then visualise again
# np.log1p(x) = log(1 + x). We use it instead of np.log because log(0) = -infinity
for col in cols_to_plot:
    plt.hist(np.log1p(tb[col]), bins=40, color='green')
    plt.title(f"log(1 + {col})")
    plt.xlabel(f"log {col}"); plt.ylabel("count")
    plt.show()
# Observe: the long right tail is squashed and the shape looks much more symmetric.

In [ ]:
# 5. Map all numeric columns to the [0, 1] interval (min-max scaling)
# scaled = (x - min) / (max - min)
tb_num = tb[numeric_cols]
col_range = (tb_num.max() - tb_num.min()).replace(0, 1)   # avoid dividing by 0 for constant columns
tb_scaled = (tb_num - tb_num.min()) / col_range

print(tb_scaled.min())   # all 0
print(tb_scaled.max())   # all 1 (0 for constant columns)

In [ ]:
# 6. Compare the means of each column
comparison = pd.DataFrame({'original mean': tb_num.mean(),
                           'scaled mean (0-1)': tb_scaled.mean()})
print(comparison)

tb_scaled.mean().sort_values().plot(kind='barh', figsize=(6, 12))
plt.title("Mean of each column after scaling to [0, 1]")
plt.show()
# Before scaling the means are impossible to compare (population is in millions,
# rates are in hundreds). After scaling they are all between 0 and 1. A low scaled
# mean means most countries sit near the column's minimum -> right-skewed column.